# Exercise 9 — Ethernet Header Parser (PYNQ-Z2)

Hardware-in-the-loop tests for the VHDL Ethernet frame buffer and header parser.

**Scope:** Python writes synthetic Ethernet II header bytes to the Zynq programmable logic (PL) through AXI GPIO. The PL stores bytes in a 2,048-byte buffer and exposes the decoded header through registers. **This notebook does not capture network traffic from the RJ45 port.**

Run the cells **from top to bottom** after programming the matching Vivado bitstream. The notebook does not load or reconfigure the FPGA itself.

## 1. AXI GPIO register map

| Peripheral | Base address | Channel 1 | Channel 2 |
| --- | --- | --- | --- |
| Command | `0x41200000` | 32-bit command (output) | — |
| Status | `0x41210000` | 32-bit status (input) | — |
| Destination | `0x41220000` | `dst_low` | `dst_high` and flags |
| Source | `0x41230000` | `src_low` | `src_high` |

Channel data offsets: `0x00` (channel 1), `0x08` (channel 2). Direction registers: `0x04`, `0x0C`.

**Command:** bits `[7:0]` data byte, `8` last, `9` send toggle, `10` ACK toggle, `[21:11]` BRAM read address, `22` read toggle.

**Status:** bits `[7:0]` read data, `[19:8]` stored size, `20` ready, `21` overflow, `22` send seen, `23` ACK seen, `24` read seen.

Each request flips a toggle and waits for the corresponding *seen* bit. The driver preserves other command fields to avoid accidental extra operations.

In [ ]:
import time
from pynq import MMIO

GPIO_LENGTH = 0x1000
DATA_OFFSET = 0x00
CH1_TRI_OFFSET = 0x04
CH2_DATA_OFFSET = 0x08
CH2_TRI_OFFSET = 0x0C

CMD_BASE = 0x41200000
STATUS_BASE = 0x41210000
DST_BASE = 0x41220000
SRC_BASE = 0x41230000
BUFFER_CAPACITY = 2048

CMD_LAST = 8
CMD_SEND_TOGGLE = 9
CMD_ACK_TOGGLE = 10
CMD_READ_ADDR_SHIFT = 11
CMD_READ_TOGGLE = 22

STS_SIZE_SHIFT = 8
STS_READY = 20
STS_OVERFLOW = 21
STS_SEND_SEEN = 22
STS_ACK_SEEN = 23
STS_READ_SEEN = 24

def get_bit(word, position):
    return (word >> position) & 1

# Configure the command GPIO as output and status/parser GPIOs as inputs.
gpio_cmd = MMIO(CMD_BASE, GPIO_LENGTH)
gpio_status = MMIO(STATUS_BASE, GPIO_LENGTH)
gpio_dst = MMIO(DST_BASE, GPIO_LENGTH)
gpio_src = MMIO(SRC_BASE, GPIO_LENGTH)

gpio_cmd.write(CH1_TRI_OFFSET, 0x00000000)
gpio_status.write(CH1_TRI_OFFSET, 0xFFFFFFFF)
for gpio in (gpio_dst, gpio_src):
    gpio.write(CH1_TRI_OFFSET, 0xFFFFFFFF)
    gpio.write(CH2_TRI_OFFSET, 0xFFFFFFFF)

print("AXI GPIO peripherals initialized.")

## 2. Reusable GPIO bridge driver

The bridge retains its 32-bit command word and synchronizes toggle bits with the hardware's acknowledgement bits. A timeout prevents a test from hanging indefinitely.

If the notebook is rerun without reprogramming the FPGA, the driver reads the existing command register rather than assuming all toggles are zero.

In [ ]:
class EthernetGpioBridge:
    def __init__(self, command_gpio, status_gpio, timeout=2.0):
        self.command_gpio = command_gpio
        self.status_gpio = status_gpio
        self.timeout = timeout
        self.command = self.command_gpio.read(DATA_OFFSET)
        status = self.status()

        # An unmatched toggle indicates an unfinished operation from a prior run.
        for cmd_bit, seen_bit in (
            (CMD_SEND_TOGGLE, STS_SEND_SEEN),
            (CMD_ACK_TOGGLE, STS_ACK_SEEN),
            (CMD_READ_TOGGLE, STS_READ_SEEN),
        ):
            if get_bit(self.command, cmd_bit) != get_bit(status, seen_bit):
                raise RuntimeError(
                    "A GPIO operation is still pending. Recheck the FPGA state "
                    "before running the tests."
                )

    def status(self):
        return self.status_gpio.read(DATA_OFFSET)

    def wait_for(self, condition, description):
        deadline = time.monotonic() + self.timeout
        while time.monotonic() < deadline:
            status = self.status()
            if condition(status):
                return status
            time.sleep(0.0001)
        raise TimeoutError(f"Timed out waiting for {description}")

    def toggle(self, cmd_bit, seen_bit):
        self.command ^= 1 << cmd_bit
        expected = get_bit(self.command, cmd_bit)
        self.command_gpio.write(DATA_OFFSET, self.command)
        return self.wait_for(
            lambda status: get_bit(status, seen_bit) == expected,
            f"toggle {cmd_bit} acknowledgement",
        )

    def send_byte(self, byte_value, last=False):
        if not 0 <= byte_value <= 0xFF:
            raise ValueError("The data byte must be between 0 and 255")
        # Replace only the byte and last-byte flag; retain all toggle bits.
        self.command = (self.command & ~0x1FF) | byte_value | (int(last) << CMD_LAST)
        self.toggle(CMD_SEND_TOGGLE, STS_SEND_SEEN)

    def send_frame(self, frame):
        if not frame:
            raise ValueError("The frame cannot be empty")
        if get_bit(self.status(), STS_READY):
            raise RuntimeError("Buffer is busy; acknowledge the previous frame first")
        for index, value in enumerate(frame):
            self.send_byte(value, last=(index == len(frame) - 1))
        return self.wait_for(
            lambda status: get_bit(status, STS_READY) == 1,
            "frame ready",
        )

    def read_byte(self, address):
        if not 0 <= address < BUFFER_CAPACITY:
            raise ValueError("BRAM address is outside the 2,048-byte buffer")
        mask = 0x7FF << CMD_READ_ADDR_SHIFT
        self.command = (self.command & ~mask) | (address << CMD_READ_ADDR_SHIFT)
        status = self.toggle(CMD_READ_TOGGLE, STS_READ_SEEN)
        return status & 0xFF

    def acknowledge(self):
        if not get_bit(self.status(), STS_READY):
            return self.status()
        self.toggle(CMD_ACK_TOGGLE, STS_ACK_SEEN)
        return self.wait_for(
            lambda status: get_bit(status, STS_READY) == 0,
            "buffer release",
        )

bridge = EthernetGpioBridge(gpio_cmd, gpio_status)

# Start from a clean buffer even if a previous notebook run left a ready frame.
bridge.acknowledge()
print("Bridge ready; buffer is idle.")

## 3. Decode the buffer status and Ethernet header

The parser's high destination register packs the flags in bits 16–19. The source-high register packs the two source-MAC bytes and the EtherType.

A completed header is not necessarily valid: for a short or overflowing frame, `header_done = 1` while `header_valid = 0`.

In [ ]:
def read_buffer_status():
    raw = bridge.status()
    return {
        "size": (raw >> STS_SIZE_SHIFT) & 0xFFF,
        "ready": get_bit(raw, STS_READY),
        "overflow": get_bit(raw, STS_OVERFLOW),
    }


def read_parser():
    dst_low = gpio_dst.read(DATA_OFFSET)
    dst_high = gpio_dst.read(CH2_DATA_OFFSET)
    src_low = gpio_src.read(DATA_OFFSET)
    src_high = gpio_src.read(CH2_DATA_OFFSET)

    dst_value = ((dst_high & 0xFFFF) << 32) | dst_low
    src_value = ((src_high & 0xFFFF) << 32) | src_low
    return {
        "dst_low": dst_low,
        "dst_high": dst_high,
        "src_low": src_low,
        "src_high": src_high,
        "destination_mac": ":".join(f"{b:02X}" for b in dst_value.to_bytes(6, "big")),
        "source_mac": ":".join(f"{b:02X}" for b in src_value.to_bytes(6, "big")),
        "ethertype": (src_high >> 16) & 0xFFFF,
        "header_done": get_bit(dst_high, 16),
        "header_valid": get_bit(dst_high, 17),
        "is_ipv4": get_bit(dst_high, 18),
        "is_arp": get_bit(dst_high, 19),
    }


def show_result(name, status, header):
    print(f"\n{name}")
    print(f"  Frame size : {status['size']}")
    print(f"  Ready      : {status['ready']}")
    print(f"  Overflow   : {status['overflow']}")
    print(f"  Destination: {header['destination_mac']}")
    print(f"  Source     : {header['source_mac']}")
    print(f"  EtherType  : 0x{header['ethertype']:04X}")
    for flag in ("header_done", "header_valid", "is_ipv4", "is_arp"):
        print(f"  {flag:12}: {header[flag]}")


def run_case(name, frame, expected, verify_bram=False):
    """Send a frame, assert the resulting flags, and release the buffer."""
    try:
        bridge.send_frame(frame)
        status = read_buffer_status()
        header = read_parser()
        show_result(name, status, header)

        actual = {**status, **header}
        for key, wanted in expected.items():
            assert actual[key] == wanted, (
                f"{name}: expected {key}={wanted!r}, got {actual[key]!r}"
            )

        if verify_bram:
            recovered = bytes(bridge.read_byte(i) for i in range(len(frame)))
            assert recovered == frame, f"{name}: BRAM readback differs from input"
            print("  BRAM readback: MATCH")

        print(f"PASS: {name}")
    finally:
        # Release the frame only after reading the parser output.
        if get_bit(bridge.status(), STS_READY):
            bridge.acknowledge()
            cleared_status = read_buffer_status()
            cleared_header = read_parser()
            assert cleared_status["ready"] == 0
            assert cleared_status["overflow"] == 0
            for key in ("dst_low", "dst_high", "src_low", "src_high"):
                assert cleared_header[key] == 0, f"{name}: {key} did not reset"
            print("  ACK/reset: PASS")

## 4. Test cases

All tests are based on the hardware results obtained for this exercise. They use synthetic frame bytes and check the actual AXI GPIO registers.

The 14-byte IPv4 and ARP examples are **Ethernet header test vectors**, not complete on-the-wire Ethernet frames (there is no payload, padding, or FCS).

In [ ]:
IPV4_HEADER = bytes.fromhex(
    "00 E0 4C 68 27 F6 76 D6 D6 82 15 66 08 00"
)
ARP_HEADER = bytes.fromhex(
    "FF FF FF FF FF FF 12 34 56 78 9A BC 08 06"
)
SHORT_HEADER = IPV4_HEADER[:10]
OVERFLOW_FRAME = IPV4_HEADER + bytes(BUFFER_CAPACITY + 1 - len(IPV4_HEADER))

assert len(IPV4_HEADER) == 14
assert len(ARP_HEADER) == 14
assert len(SHORT_HEADER) == 10
assert len(OVERFLOW_FRAME) == BUFFER_CAPACITY + 1

### Test A — IPv4 header and BRAM readback

Checks both the parser output and byte-for-byte retrieval of the header from BRAM.

In [ ]:
run_case(
    "IPv4 header",
    IPV4_HEADER,
    {
        "size": 14, "ready": 1, "overflow": 0,
        "dst_low": 0x4C6827F6, "dst_high": 0x000700E0,
        "src_low": 0xD6821566, "src_high": 0x080076D6,
        "header_done": 1, "header_valid": 1, "is_ipv4": 1, "is_arp": 0,
    },
    verify_bram=True,
)

### Test B — ARP EtherType

Checks that EtherType `0x0806` selects ARP rather than IPv4.

In [ ]:
run_case(
    "ARP header",
    ARP_HEADER,
    {
        "size": 14, "ready": 1, "overflow": 0,
        "dst_low": 0xFFFFFFFF, "dst_high": 0x000BFFFF,
        "src_low": 0x56789ABC, "src_high": 0x08061234,
        "header_done": 1, "header_valid": 1, "is_ipv4": 0, "is_arp": 1,
    },
)

### Test C — Truncated header

Ten bytes are not enough for the 14-byte Ethernet II header. `header_done` indicates that the frame ended, while `header_valid` must stay clear.

In [ ]:
run_case(
    "Truncated header",
    SHORT_HEADER,
    {
        "size": 10, "ready": 1, "overflow": 0,
        "header_done": 1, "header_valid": 0, "is_ipv4": 0, "is_arp": 0,
    },
)

### Test D — BRAM overflow

Send **2,049 bytes** into a **2,048-byte** buffer. The buffer must report overflow and clamp the stored size to 2,048. The parser must invalidate the header even though the first 14 bytes look like IPv4.

This test sends each byte through AXI GPIO and is therefore slower than the 14-byte cases.

In [ ]:
run_case(
    "BRAM overflow",
    OVERFLOW_FRAME,
    {
        "size": BUFFER_CAPACITY, "ready": 1, "overflow": 1,
        "header_done": 1, "header_valid": 0, "is_ipv4": 0, "is_arp": 0,
    },
)
print("\nAll four hardware test cases passed.")

## 5. Live Ethernet capture and FPGA validation (optional)

Capture a **real incoming Ethernet II frame** from `eth0` via the Linux `AF_PACKET` socket API.

Data path: **RJ45 → PS GEM → Linux AF_PACKET → Python → AXI GPIO → PL BRAM/parser**.
The PL does not directly receive the Ethernet PHY interface in this design.

- This step requires Linux `CAP_NET_RAW` permission (often root) and incoming traffic on `eth0`.
- Set `RUN_LIVE_CAPTURE = True` to perform it. It is disabled by default so **Run All** is deterministic and does not wait for network traffic.
- The test skips malformed/short headers and frames larger than the 2,048-byte buffer.
- Linux packet sockets normally expose Ethernet frames without the PHY preamble and FCS.
- Generate incoming traffic from a different device if necessary, such as a ping to the PYNQ-Z2.


In [ ]:
import socket

RUN_LIVE_CAPTURE = False
CAPTURE_INTERFACE = "eth0"
CAPTURE_TIMEOUT_SECONDS = 15


def capture_ethernet_frame(interface="eth0", timeout=15, max_bytes=2048):
    """Return a valid-sized incoming Ethernet frame from a Linux network interface."""
    protocol_all = 0x0003  # ETH_P_ALL
    packet_outgoing = getattr(socket, "PACKET_OUTGOING", 4)
    deadline = time.monotonic() + timeout

    with socket.socket(socket.AF_PACKET, socket.SOCK_RAW,
                       socket.htons(protocol_all)) as sock:
        sock.bind((interface, 0))
        while True:
            remaining = deadline - time.monotonic()
            if remaining <= 0:
                raise TimeoutError(f"No suitable incoming frame on {interface}")
            sock.settimeout(remaining)
            try:
                frame, address = sock.recvfrom(65535)
            except socket.timeout as exc:
                raise TimeoutError(f"No suitable incoming frame on {interface}") from exc

            # AF_PACKET sockaddr_ll: (interface, protocol, packet_type, ...)
            if address[2] == packet_outgoing:
                continue
            if 14 <= len(frame) <= max_bytes:
                return bytes(frame)


def validate_real_frame(frame):
    """Compare the actual Ethernet bytes with PL header and BRAM readback."""
    if not 14 <= len(frame) <= BUFFER_CAPACITY:
        raise ValueError("Frame must be between 14 and 2048 bytes")

    expected_dst = ":".join(f"{b:02X}" for b in frame[:6])
    expected_src = ":".join(f"{b:02X}" for b in frame[6:12])
    expected_type = int.from_bytes(frame[12:14], "big")

    try:
        bridge.send_frame(frame)
        status = read_buffer_status()
        result = read_parser()
        show_result("Live Ethernet frame", status, result)

        assert status == {"size": len(frame), "ready": 1, "overflow": 0}
        assert result["header_done"] == 1
        assert result["header_valid"] == 1
        assert result["destination_mac"] == expected_dst
        assert result["source_mac"] == expected_src
        assert result["ethertype"] == expected_type
        assert result["is_ipv4"] == int(expected_type == 0x0800)
        assert result["is_arp"] == int(expected_type == 0x0806)

        recovered = bytes(bridge.read_byte(i) for i in range(len(frame)))
        assert recovered == frame, "BRAM readback differs from captured frame"
        print("PASS: Real Ethernet header and BRAM match the Linux capture")
        return result
    finally:
        if get_bit(bridge.status(), STS_READY):
            bridge.acknowledge()
            assert read_buffer_status()["ready"] == 0
            assert read_buffer_status()["overflow"] == 0
            assert read_parser()["header_done"] == 0
            print("  ACK/reset: PASS")


In [ ]:
if RUN_LIVE_CAPTURE:
    print(f"Waiting for an incoming Ethernet frame on {CAPTURE_INTERFACE}...")
    live_frame = capture_ethernet_frame(
        CAPTURE_INTERFACE,
        timeout=CAPTURE_TIMEOUT_SECONDS,
        max_bytes=BUFFER_CAPACITY,
    )
    print(f"Captured {len(live_frame)} bytes")
    print("Raw Ethernet header:", live_frame[:14].hex(" ").upper())
    live_result = validate_real_frame(live_frame)
else:
    print("Live capture skipped. Set RUN_LIVE_CAPTURE = True to enable it.")


## 6. Interpretation and limitations

- Synthetic tests validate the bridge, buffer, header extraction, overflow handling, and ACK/reset.
- The optional live test validates a **real Ethernet frame captured by Linux and forwarded to the PL**.
- The transfer uses an AXI GPIO software-driven byte stream, not a direct GEM-to-PL connection or AXI DMA.
- Only the Ethernet II destination MAC, source MAC and EtherType are parsed; no IPv4/TCP payload validation is performed.
- Running the notebook on a different hardware design requires verifying the GPIO base addresses and bit definitions.
